# A/H price movement from H-share IPO — China semis

Nine A+H names. Each is anchored on **its own H-share listing date**, and every chart
runs continuously from **day 0 to 2 months** off that date.

The x-axis is *calendar days since listing*, not trading days — Shanghai/Shenzhen and
HKEX have different holidays, so a trading-day count would drift the two legs onto
different wall-clock dates and corrupt the premium.

Run the cells in order. Everything renders inline; nothing is written to Excel.

| Company | A-share | H-share | H IPO |
|---|---|---|---|
| CFMEE 芯碁微装 | 688630 CH | 9630 HK | 2026-06-26 |
| SG Micro 圣邦股份 | 300661 CH | 3661 HK | 2026-06-26 |
| GigaDevice 兆易创新 | 603986 CH | 3986 HK | 2026-01-13 |
| OmniVision 豪威集团 | 603501 CH | 501 HK | 2026-01-12 |
| NOVOSENSE 纳芯微 | 688052 CH | 2676 HK | 2025-12-08 |
| Fortior 峰岹科技 | 688279 CH | 1304 HK | 2025-07-09 |
| SICC 天岳先进 | 688234 CH | 2631 HK | 2025-08-20 |
| Nexchip 晶合集成 | 688249 CH | 2249 HK | 2026-07-10 |
| Montage Tech 澜起科技 | 688008 CH | 6809 HK | 2026-02-09 |

CFMEE, SG Micro and Nexchip are less than 2 months old as of today, so their lines
simply stop short. That is correct behaviour, not missing data.


## 1. Setup

In [ ]:
%matplotlib inline

import datetime as dt
import os
from dataclasses import dataclass
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta
from matplotlib import font_manager

# ---- CJK font -------------------------------------------------------------
# matplotlib's default font has no Chinese glyphs, which is what produces the
# "Glyph ... missing from current font" warnings.
def setup_fonts() -> bool:
    for c in ["Microsoft YaHei", "Microsoft JhengHei", "SimHei", "SimSun",
              "PingFang SC", "Hiragino Sans GB", "Arial Unicode MS",
              "Noto Sans CJK SC", "Noto Sans SC", "WenQuanYi Zen Hei"]:
        if c in {f.name for f in font_manager.fontManager.ttflist}:
            plt.rcParams["font.sans-serif"] = [c, "DejaVu Sans"]
            plt.rcParams["axes.unicode_minus"] = False
            print(f"CJK font: {c}")
            return True
    print("No CJK font found; Chinese names will be dropped from labels.\n"
          "  If you expect Microsoft YaHei, rebuild the cache with:\n"
          "    font_manager._load_fontmanager(try_read_cache=False)")
    return False

HAS_CJK = setup_fonts()

plt.rcParams.update({
    "font.size": 9, "figure.dpi": 110, "figure.facecolor": "white",
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
})

A_COL, H_COL, P_COL = "#c0392b", "#1f4e79", "#7d6608"

# ---- config ---------------------------------------------------------------
BBG_HOST, BBG_PORT = "localhost", 8194
FIELD      = "PX_LAST"          # or TOT_RETURN_INDEX_GROSS_DVDS
FX_TICKER  = "CNYHKD Curncy"    # HKD per 1 CNY -> A_hkd = A_cny * CNYHKD
MONTHS     = 2                  # length of the event window
CACHE      = "px_cache.parquet" # price cache so you only hit Bloomberg once

# guide lines drawn on every chart, in calendar days since listing
GUIDES = {"1w": 7, "2w": 14, "1M": 30, "2M": 61}


@dataclass
class Pair:
    name: str
    cn_name: str
    a_tkr: str
    h_tkr: str
    ipo: str
    offer_px: Optional[float] = None

    @property
    def t0(self) -> pd.Timestamp:
        return pd.Timestamp(self.ipo)

    @property
    def label(self) -> str:
        return f"{self.name} ({self.cn_name})" if HAS_CJK else self.name


# Verified against each issuer's H-share listing announcement.
# Bloomberg HK tickers carry no leading zeros: 0501.HK -> "501 HK Equity".
PAIRS = [
    Pair("CFMEE",        "芯碁微装", "688630 CH Equity", "9630 HK Equity", "2026-06-26", 252.73),
    Pair("SG Micro",     "圣邦股份", "300661 CH Equity", "3661 HK Equity", "2026-06-26",  85.20),
    Pair("GigaDevice",   "兆易创新", "603986 CH Equity", "3986 HK Equity", "2026-01-13", 162.00),
    Pair("OmniVision",   "豪威集团", "603501 CH Equity",  "501 HK Equity", "2026-01-12",   None),
    Pair("NOVOSENSE",    "纳芯微",   "688052 CH Equity", "2676 HK Equity", "2025-12-08", 116.00),
    Pair("Fortior",      "峰岹科技", "688279 CH Equity", "1304 HK Equity", "2025-07-09", 120.50),
    Pair("SICC",         "天岳先进", "688234 CH Equity", "2631 HK Equity", "2025-08-20",  42.80),
    Pair("Nexchip",      "晶合集成", "688249 CH Equity", "2249 HK Equity", "2026-07-10",  32.30),
    Pair("Montage Tech", "澜起科技", "688008 CH Equity", "6809 HK Equity", "2026-02-09", 106.89),
]

print(f"{len(PAIRS)} pairs configured")

## 2. Pull from Bloomberg

One `HistoricalDataRequest` covering every leg plus FX. The start date is the
earliest IPO less a small buffer, so a single pull serves all nine windows.

Set `offline=True` on later runs to re-chart from the cached parquet without a
terminal connection.

In [ ]:
def fetch_history(securities, fields, start, end, chunk=10):
    """blpapi HistoricalDataRequest -> long DataFrame [date, security, field, value]."""
    import blpapi

    opts = blpapi.SessionOptions()
    opts.setServerHost(BBG_HOST)
    opts.setServerPort(BBG_PORT)
    session = blpapi.Session(opts)
    if not session.start():
        raise RuntimeError("Could not start blpapi session. Is the terminal running?")

    try:
        if not session.openService("//blp/refdata"):
            raise RuntimeError("Could not open //blp/refdata")
        svc = session.getService("//blp/refdata")

        rows = []
        for i in range(0, len(securities), chunk):
            batch = securities[i:i + chunk]
            req = svc.createRequest("HistoricalDataRequest")
            for s in batch:
                req.getElement("securities").appendValue(s)
            for f in fields:
                req.getElement("fields").appendValue(f)
            req.set("startDate", start.strftime("%Y%m%d"))
            req.set("endDate", end.strftime("%Y%m%d"))
            req.set("periodicitySelection", "DAILY")
            req.set("periodicityAdjustment", "ACTUAL")
            req.set("nonTradingDayFillOption", "ACTIVE_DAYS_ONLY")
            req.set("adjustmentSplit", True)      # splits on, dividends off, so the
            req.set("adjustmentAbnormal", False)  # premium stays a clean price ratio
            req.set("adjustmentNormal", False)

            print(f"  requesting {len(batch)} securities ...")
            session.sendRequest(req)

            done = False
            while not done:
                ev = session.nextEvent(30_000)
                for msg in ev:
                    if msg.hasElement("responseError"):
                        raise RuntimeError(msg.getElement("responseError"))
                    if not msg.hasElement("securityData"):
                        continue
                    sd = msg.getElement("securityData")
                    sec = sd.getElementAsString("security")
                    if sd.hasElement("securityError"):
                        print(f"  !! {sec}: {sd.getElement('securityError')}")
                        continue
                    if sd.hasElement("fieldExceptions"):
                        fx = sd.getElement("fieldExceptions")
                        for j in range(fx.numValues()):
                            print(f"  !! {sec}: {fx.getValue(j)}")
                    fd = sd.getElement("fieldData")
                    for j in range(fd.numValues()):
                        pt = fd.getValue(j)
                        d = pt.getElementAsDatetime("date")
                        for f in fields:
                            if pt.hasElement(f):
                                rows.append((pd.Timestamp(d), sec, f,
                                             pt.getElementAsFloat(f)))
                if ev.eventType() == blpapi.Event.RESPONSE:
                    done = True
        return pd.DataFrame(rows, columns=["date", "security", "field", "value"])
    finally:
        session.stop()


def pull(offline=False, end=None, field=FIELD):
    """Wide price panel: index = date, columns = security."""
    end = pd.Timestamp(end).date() if end else dt.date.today()
    start = (min(p.t0 for p in PAIRS) - pd.Timedelta(days=30)).date()

    if offline:
        if not os.path.exists(CACHE):
            raise FileNotFoundError(f"No cache at {CACHE}; run pull() first.")
        long_df = pd.read_parquet(CACHE)
        print(f"Loaded cache: {len(long_df):,} rows")
    else:
        secs = [p.a_tkr for p in PAIRS] + [p.h_tkr for p in PAIRS] + [FX_TICKER]
        print(f"Pulling {len(secs)} securities, {start} -> {end}")
        long_df = fetch_history(secs, [field], start, end)
        if long_df.empty:
            raise RuntimeError("Bloomberg returned nothing. Check tickers / entitlements.")
        long_df.to_parquet(CACHE)
        print(f"Cached {len(long_df):,} rows -> {CACHE}")

    sub = long_df[long_df["field"] == field]
    return sub.pivot_table(index="date", columns="security", values="value").sort_index()


px = pull()          # <- pull(offline=True) to reuse the cache
px.tail(3)

## 3. Build each stock's own event window

Every series is re-indexed onto **calendar days since that stock's listing date**,
then clipped at 2 months. Day 0 is the listing day close, which is the common base
for both legs — the A leg has no offer price, so anchoring on the offer would make
the two legs non-comparable. The IPO pop is kept as a separate number instead.

In [ ]:
def to_days(s: pd.Series, t0: pd.Timestamp) -> pd.Series:
    """Re-index a dated series onto integer calendar days since t0."""
    out = s.copy()
    out.index = (s.index - t0).days
    return out


def build_window(pair: Pair, wide: pd.DataFrame, fx: pd.Series, months=MONTHS) -> dict:
    t0  = pair.t0
    end = t0 + relativedelta(months=months)

    a = wide[pair.a_tkr].dropna()
    h = wide[pair.h_tkr].dropna()
    a = a[(a.index >= t0) & (a.index <= end)]
    h = h[(h.index >= t0) & (h.index <= end)]
    if a.empty or h.empty:
        return {}

    a0, h0 = float(a.iloc[0]), float(h.iloc[0])

    # A leg converted to HKD so the two are directly comparable
    a_hkd = a * fx.reindex(a.index).ffill()

    # premium only on dates when BOTH markets traded
    common = a.index.intersection(h.index)
    prem = ((a_hkd.reindex(common) / h.reindex(common)) - 1.0) * 100.0
    prem = prem.dropna()

    return {
        "pair":   pair,
        "t0":     t0,
        "max_day": int((min(end, max(a.index[-1], h.index[-1])) - t0).days),
        "a":      to_days(a, t0),               # CNY
        "h":      to_days(h, t0),               # HKD
        "a_hkd":  to_days(a_hkd, t0),           # A leg in HKD
        "a_reb":  to_days(a / a0 * 100, t0),
        "h_reb":  to_days(h / h0 * 100, t0),
        "prem":   to_days(prem, t0),
        "a0": a0, "h0": h0,
        "pop": (h0 / pair.offer_px - 1) * 100 if pair.offer_px else np.nan,
    }


fx = px[FX_TICKER].ffill()
W = {}
for p in PAIRS:
    if p.a_tkr not in px.columns or p.h_tkr not in px.columns:
        print(f"  skip {p.name}: a leg is missing from the pull")
        continue
    w = build_window(p, px, fx)
    if w:
        W[p.name] = w
    else:
        print(f"  skip {p.name}: no prices at/after {p.ipo}")

for n, w in W.items():
    print(f"{n:<13} t0={w['t0'].date()}  window covers day 0 -> {w['max_day']}")

## 4. Summary table

Discrete read-offs from the same continuous series, for reference. Blank cells are
horizons the stock has not reached yet.

In [ ]:
def val_at(s: pd.Series, day: int, tol: int = 4):
    """Last observation at or before `day`. NaN if `day` is past the sample,
    so unreached horizons stay blank rather than snapping to the latest print."""
    sub = s[s.index <= day]
    if sub.empty or day > s.index.max() + tol:
        return np.nan
    return float(sub.iloc[-1])


rows = []
for name, w in W.items():
    p = w["pair"]
    # 1M/2M resolved as true calendar months off this stock's own t0
    days = {"1d": 1, "1w": 7, "2w": 14,
            "1M": (p.t0 + relativedelta(months=1) - p.t0).days,
            "2M": (p.t0 + relativedelta(months=2) - p.t0).days}

    r = {"Name": name, "IPO": p.t0.date(),
         "Offer": p.offer_px, "Day0 close": round(w["h0"], 2),
         "Pop %": round(w["pop"], 1) if w["pop"] == w["pop"] else np.nan,
         "Prem d0 %": round(float(w["prem"].iloc[0]), 1) if len(w["prem"]) else np.nan}

    for lab, d in days.items():
        a1, h1 = val_at(w["a_reb"], d), val_at(w["h_reb"], d)
        pr = val_at(w["prem"], d)
        r[f"A {lab}"] = round(a1 - 100, 1) if a1 == a1 else np.nan
        r[f"H {lab}"] = round(h1 - 100, 1) if h1 == h1 else np.nan
        r[f"Prem {lab}"] = round(pr, 1) if pr == pr else np.nan
    rows.append(r)

summary = pd.DataFrame(rows)

pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 60)

display(summary[["Name", "IPO", "Offer", "Day0 close", "Pop %"]
                + [f"{leg} {l}" for l in ["1d","1w","2w","1M","2M"] for leg in ("A","H")]]
        .style.format(precision=1, na_rep="-")
        .background_gradient(cmap="RdYlGn", subset=[f"{leg} {l}"
             for l in ["1d","1w","2w","1M","2M"] for leg in ("A","H")])
        .set_caption("Return vs listing-day close, %"))

display(summary[["Name", "Prem d0 %"] + [f"Prem {l}" for l in ["1d","1w","2w","1M","2M"]]]
        .style.format(precision=1, na_rep="-")
        .background_gradient(cmap="RdYlGn_r")
        .set_caption("A/H premium, % (positive = A trades above H)"))

## 5. Price movement, one figure per stock

Three panels, all sharing a day-0-to-2-month x-axis:

1. **Native price levels** — A in CNY on the left axis, H in HKD on the right.
   Twin axes because the units differ.
2. **Common currency** — both legs in HKD on one axis, with the gap shaded. The
   shaded distance *is* the A/H premium, which reads more naturally than the
   premium line when you're judging whether a spread is converging.
3. **A/H premium** — the same thing as a percentage.

In [ ]:
def guides(ax, max_day):
    for lab, d in GUIDES.items():
        if d <= max_day + 2:
            ax.axvline(d, color="0.7", lw=0.7, ls=":", zorder=0)
            ax.annotate(lab, xy=(d, 1), xycoords=("data", "axes fraction"),
                        xytext=(2, -10), textcoords="offset points",
                        fontsize=7, color="0.45")


def plot_stock(w: dict):
    p, md = w["pair"], w["max_day"]
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4.2),
                                        constrained_layout=True)

    # -- 1. native price levels, twin axes ---------------------------------
    ax1.plot(w["a"].index, w["a"].values, color=A_COL, lw=1.7)
    ax1.set_ylabel("A share, CNY", color=A_COL)
    ax1.tick_params(axis="y", labelcolor=A_COL)
    ax1b = ax1.twinx()
    ax1b.plot(w["h"].index, w["h"].values, color=H_COL, lw=1.7)
    ax1b.set_ylabel("H share, HKD", color=H_COL)
    ax1b.tick_params(axis="y", labelcolor=H_COL)
    ax1b.grid(False)
    ax1b.spines["right"].set_visible(True)
    ax1.set_title("Price levels, native currency", loc="left", fontsize=9.5)
    guides(ax1, md)

    # -- 2. both legs in HKD, gap shaded -----------------------------------
    ax2.plot(w["a_hkd"].index, w["a_hkd"].values, color=A_COL, lw=1.7,
             label="A, in HKD")
    ax2.plot(w["h"].index, w["h"].values, color=H_COL, lw=1.7, label="H, HKD")
    common = w["a_hkd"].index.intersection(w["h"].index)
    if len(common):
        av, hv = w["a_hkd"].reindex(common).values, w["h"].reindex(common).values
        ax2.fill_between(common, av, hv, where=av >= hv, color=A_COL,
                         alpha=0.13, interpolate=True)
        ax2.fill_between(common, av, hv, where=av < hv, color=H_COL,
                         alpha=0.13, interpolate=True)
    ax2.set_ylabel("HKD")
    ax2.legend(frameon=False, fontsize=8)
    ax2.set_title("Common currency, shaded gap = premium", loc="left", fontsize=9.5)
    guides(ax2, md)

    # -- 3. premium ---------------------------------------------------------
    pr = w["prem"]
    ax3.plot(pr.index, pr.values, color=P_COL, lw=1.7)
    ax3.axhline(0, color="0.4", lw=0.8)
    if len(pr):
        ax3.fill_between(pr.index, 0, pr.values, where=pr.values >= 0,
                         color=A_COL, alpha=0.15)
        ax3.fill_between(pr.index, 0, pr.values, where=pr.values < 0,
                         color=H_COL, alpha=0.15)
    ax3.set_ylabel("A/H premium, %")
    ax3.set_title("Premium (positive = A above H)", loc="left", fontsize=9.5)
    guides(ax3, md)

    for ax in (ax1, ax2, ax3):
        ax.set_xlim(0, max(md, 5))
        ax.set_xlabel("Calendar days since listing")

    pop = f"   |   day-1 pop {w['pop']:+.0f}%" if w["pop"] == w["pop"] else ""
    fig.suptitle(f"{p.label}   |   {p.a_tkr.split()[0]} / {p.h_tkr.split()[0]}"
                 f"   |   listed {w['t0'].date()}{pop}",
                 fontweight="bold", fontsize=11.5, x=0.005, ha="left")
    plt.show()


for name in W:
    plot_stock(W[name])

## 6. Cross-sectional overlays

All nine on one set of axes in event time, so the different listing dates line up
at day 0.

In [ ]:
cmap = plt.get_cmap("tab10")

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4.6), constrained_layout=True)
for i, (name, w) in enumerate(W.items()):
    c = cmap(i % 10)
    ax1.plot(w["a_reb"].index, w["a_reb"].values, lw=1.4, color=c, label=name)
    ax2.plot(w["h_reb"].index, w["h_reb"].values, lw=1.4, color=c, label=name)
    ax3.plot(w["prem"].index,  w["prem"].values,  lw=1.4, color=c, label=name)

for ax, ttl, base in ((ax1, "A share, rebased", 100),
                      (ax2, "H share, rebased", 100),
                      (ax3, "A/H premium, %", 0)):
    ax.axhline(base, color="0.4", lw=0.8)
    ax.set_title(ttl, loc="left", fontweight="bold")
    ax.set_xlabel("Calendar days since listing")
    ax.set_xlim(0, 61)
    for d in GUIDES.values():
        ax.axvline(d, color="0.85", lw=0.6, ls=":", zorder=0)

ax1.set_ylabel("Listing-day close = 100")
ax3.set_ylabel("%")
ax3.legend(frameon=False, fontsize=7, ncol=2)
fig.suptitle("Day 0 to 2 months, every name anchored on its own H-share listing date",
             fontweight="bold", fontsize=11.5, x=0.005, ha="left")
plt.show()


# small multiples: A vs H rebased, one panel per name
n = len(W)
nrow = int(np.ceil(n / 3))
fig, axes = plt.subplots(nrow, 3, figsize=(14, 3.2 * nrow), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

for ax, (name, w) in zip(axes, W.items()):
    ax.plot(w["a_reb"].index, w["a_reb"].values, color=A_COL, lw=1.5, label="A")
    ax.plot(w["h_reb"].index, w["h_reb"].values, color=H_COL, lw=1.5, label="H")
    ax.axhline(100, color="0.4", lw=0.8)
    ax.set_xlim(0, max(w["max_day"], 5))
    ax.set_title(f"{name}   ({w['t0'].date()})", loc="left", fontsize=9)
    ax.set_xlabel("Days since listing", fontsize=8)
    ax.tick_params(labelsize=8)
for ax in axes[n:]:
    ax.set_visible(False)
axes[0].legend(frameon=False, fontsize=8)

fig.suptitle("A (red) vs H (blue), rebased to listing-day close",
             fontweight="bold", fontsize=11.5, x=0.005, ha="left")
plt.show()